# 2. Clean and split

data/interim/parsed.jsonl stays as it is.

**Clean**
- drop empty text, and text shorter than 500 characters
- keep rows with no bijzondere kenmerken; the bijzondere-kenmerken model will skip them later
- write data/processed/cleaned.jsonl

**Split**
- 
atural_test stays together
- alanced is cut 80 / 10 / 10 inside each year × area cell
- seed 42; ECLI lists go to splits.json
- rare labels are counted on **train** only and written to labels.json

In [1]:
from pathlib import Path
import json
import pandas as pd

PARSED = Path('../data/interim/parsed.jsonl')
OUT_DIR = Path('../data/processed')
MIN_TEXT_LENGTH = 500
RARE_BELOW = 100
SEED = 42

df = pd.read_json(PARSED, lines=True)
print('before:', len(df))

before: 65480


Drop empty and very short texts.

In [2]:
short = df['text_length'] < MIN_TEXT_LENGTH
print('dropped, shorter than', MIN_TEXT_LENGTH, ':', int(short.sum()))

clean = df.loc[~short].copy()
print('after:', len(clean))
print(clean['text_source'].value_counts())

dropped, shorter than 500 : 164
after: 65316
text_source
uitspraak    62321
conclusie     2995
Name: count, dtype: int64


In [3]:
clean['n_rg'] = clean['rechtsgebieden'].map(len)
clean['n_bk'] = clean['bijzondere_kenmerken'].map(len)
print('no rechtsgebieden:', int((clean['n_rg'] == 0).sum()))
print('no bijzondere kenmerken:', int((clean['n_bk'] == 0).sum()), '(kept)')

no rechtsgebieden: 0
no bijzondere kenmerken: 3007 (kept)


Hold out the natural set. Split only the balanced rows.

In [4]:
natural = clean[clean['pool'] == 'natural_test'].copy()
balanced = clean[clean['pool'] == 'balanced'].copy()
print('natural_test:', len(natural))
print('balanced:', len(balanced))

trains: list[pd.DataFrame] = []
vals: list[pd.DataFrame] = []
tests: list[pd.DataFrame] = []

for _, group in balanced.groupby(['year', 'area'], sort=True):
    g = pd.DataFrame(group).sample(frac=1, random_state=SEED)
    n = len(g)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    trains.append(g.iloc[:n_train])
    vals.append(g.iloc[n_train:n_train + n_val])
    tests.append(g.iloc[n_train + n_val:])

train = pd.concat(trains, ignore_index=True).sample(frac=1, random_state=SEED)
val = pd.concat(vals, ignore_index=True).sample(frac=1, random_state=SEED)
test = pd.concat(tests, ignore_index=True).sample(frac=1, random_state=SEED)

print('train:', len(train), 'val:', len(val), 'test:', len(test))
print('sum:', len(train) + len(val) + len(test))
print()
print('train year x area')
print(pd.crosstab(train['year'], train['area']))

natural_test: 4985
balanced: 60331


train: 48251 val: 6012 test: 6068
sum: 60331

train year x area
area  Bestuursrecht  Civiel recht  Internationaal publiekrecht  Strafrecht
year                                                                      
2006            800           798                            0         799
2007            799           800                            0         800
2008            799           799                            0         800
2009            796           792                            0         799
2010            773           800                            0         800
2011            773           799                            0         800
2012            760           800                            0         800
2013            784           799                            1         799
2014            800           800                           26         797
2015            800           800                           32         796
2016            800           800   

Rare labels, counted on train only. They are listed, not stripped off the rows.

In [5]:
rg = train.explode('rechtsgebieden')['rechtsgebieden'].value_counts()
bk_train = train[train['bijzondere_kenmerken'].map(len) > 0]
bk = bk_train.explode('bijzondere_kenmerken')['bijzondere_kenmerken'].value_counts()

rg_rare = rg[rg < RARE_BELOW]
bk_rare = bk[bk < RARE_BELOW]

print('train rows with no bijzondere kenmerken:', int((train['bijzondere_kenmerken'].map(len) == 0).sum()))
print()
print('rechtsgebieden:', len(rg), 'keep:', int((rg >= RARE_BELOW).sum()), 'rare:', len(rg_rare))
print(rg_rare)
print()
print('bijzondere kenmerken:', len(bk), 'keep:', int((bk >= RARE_BELOW).sum()), 'rare:', len(bk_rare))
print(bk_rare)

train rows with no bijzondere kenmerken: 2271

rechtsgebieden: 31 keep: 20 rare: 11
rechtsgebieden
Internationaal strafrecht       96
Intellectueel-eigendomsrecht    79
Goederenrecht                   39
Aanbestedingsrecht              37
Internationaal privaatrecht     34
Penitentiair strafrecht         28
Europees civiel recht           19
Europees bestuursrecht          16
Volkenrecht                      6
Mensenrechten                    5
Mededingingsrecht                2
Name: count, dtype: int64

bijzondere kenmerken: 43 keep: 25 rare: 18
bijzondere_kenmerken
Verstek                                 80
Tussenbeschikking                       55
Verwijzing na Hoge Raad                 52
Schadevergoedingsuitspraak              52
Verschoning                             17
Prejudicieel verzoek                    13
Cassatie in het belang der wet          10
Conservatoire maatregel                  7
Versnelde behandeling                    5
Geheimhoudingsbeslissing              

Write the cleaned table, the four splits, the ECLI lists, and the label lists.

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

def write_jsonl(path: Path, frame: pd.DataFrame) -> None:
    rows = frame.drop(columns=['n_rg', 'n_bk'], errors='ignore').to_dict(orient='records')
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            row['date'] = str(row['date'])[:10]
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(path.name, len(rows))

write_jsonl(OUT_DIR / 'cleaned.jsonl', clean)
write_jsonl(OUT_DIR / 'train.jsonl', train)
write_jsonl(OUT_DIR / 'val.jsonl', val)
write_jsonl(OUT_DIR / 'test.jsonl', test)
write_jsonl(OUT_DIR / 'natural_test.jsonl', natural)

splits = {
    'seed': SEED,
    'train': train['ecli'].tolist(),
    'val': val['ecli'].tolist(),
    'test': test['ecli'].tolist(),
    'natural_test': natural['ecli'].tolist(),
}
(OUT_DIR / 'splits.json').write_text(
    json.dumps(splits, ensure_ascii=False, indent=2), encoding='utf-8'
)

labels = {
    'rare_below': RARE_BELOW,
    'counted_on': 'train',
    'rechtsgebieden_keep': list(rg[rg >= RARE_BELOW].index),
    'rechtsgebieden_rare': list(rg_rare.index),
    'bijzondere_kenmerken_keep': list(bk[bk >= RARE_BELOW].index),
    'bijzondere_kenmerken_rare': list(bk_rare.index),
}
(OUT_DIR / 'labels.json').write_text(
    json.dumps(labels, ensure_ascii=False, indent=2), encoding='utf-8'
)

notes = {
    'source': str(PARSED.as_posix()),
    'min_text_length': MIN_TEXT_LENGTH,
    'rows_before': int(len(df)),
    'rows_after': int(len(clean)),
    'rows_dropped_short': int(short.sum()),
}
(OUT_DIR / 'clean_notes.json').write_text(
    json.dumps(notes, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('wrote splits.json, labels.json, clean_notes.json')

cleaned.jsonl 65316


train.jsonl 48251


val.jsonl 6012


test.jsonl 6068


natural_test.jsonl 4985
wrote splits.json, labels.json, clean_notes.json
